## What is the CCDC (CSD) Python API, and where does GOLD fit?

The **CCDC CSD Python API** is a toolkit for working with small molecules, proteins, and crystallographic data in Python.

- **Read / write structures**: e.g., MOL2/SDF/PDB-like formats via `ccdc.io`.
- **Protein handling**: `ccdc.protein.Protein` provides common preparation operations (removing waters, adding hydrogens, cleaning unusual atoms, handling ligands).
- **Docking setup**: `ccdc.docking` provides access to **GOLD** docking *settings* and helper classes.

**GOLD** (Genetic Optimisation for Ligand Docking) is CCDC’s docking engine. In these tutorials we use the Python API primarily to:

- **Prepare the receptor** in a consistent way (so docking is reproducible).
- **Define a binding site** (the region GOLD will search), typically from a known ligand or a residue.

## Goal of this notebook

- Load a protein structure.
- Perform basic **protein preparation**:
  - remove waters and unknown atoms
  - add hydrogens
  - remove bound ligands (optional, depending on your protocol)
- Define a **binding site** in two common ways:
  - from a reference (native) ligand
  - from a residue (fallback / alternative)
- Write a prepared structure (`protein_prepared.mol2`) for use in the GOLD docking workflow (Tutorial 5).

**For more indormation:** Please visit [the documentation](https://downloads.ccdc.cam.ac.uk/documentation/API/index.html)

In [ ]:
import workshop_setup
import os

In [ ]:
from ccdc.docking import Docker
from ccdc.io import MoleculeReader, EntryReader, EntryWriter, MoleculeWriter
from ccdc.protein import Protein
from pathlib import Path
from ccdc.molecule import Molecule


In [ ]:
protein_file = './data/protein_ligand_4LQM_CHEMBL674017.mol2' #Please change this if you want to apply for another protein-ligand complex
native_ligand_file = './data/4lqm_ligand.mol2' #Please change this if you want to apply for another protein-ligand complex

In [ ]:
out_folder = './data'
protein = Protein.from_file(protein_file)
protein_id = protein.identifier
print(f'Protein {protein_id} loaded with {len(protein.ligands)} ligand(s) and {len(protein.chains)} chain(s).')
protein.remove_all_waters()
protein.remove_unknown_atoms()
protein.add_hydrogens()
for lig in protein.ligands:
    protein.remove_ligand(lig.identifier)
prep_protein_file =  os.path.join(out_folder, 'protein_prepared.mol2')
with EntryWriter(prep_protein_file) as writer:
    writer.write(protein)


### Defining the binding site (where GOLD will search)

GOLD needs a **binding site definition**: a set of atoms/residues (and an enclosing radius) that defines the region to explore during docking.

Two common strategies:

- **From a reference ligand (recommended when available)**
  - Use a known binder or co-crystallized ligand to define the pocket.
  - Typically the most reliable way to center the site.

- **From protein features (alternative)**
  - Define the site around a particular **residue**, **atom**, or **3D point**.
  - Useful when no ligand is available, or when you want to target a specific region.

In the next cells we’ll create a `Docker`, access its `settings`, then set `settings.binding_site` using each approach and print how many atoms/residues/etc. are included.

In [ ]:
#From the native ligand
native_ligand = MoleculeReader(native_ligand_file)[0]
docker = Docker()

settings = docker.settings
settings.binding_site = settings.BindingSiteFromLigand(protein, native_ligand, 10.0)

atoms = settings.binding_site.atoms
print(f'Number of atoms in the binding site: {len(atoms)}')
residues = settings.binding_site.residues
print(f'Number of residues in the binding site: {len(residues)}')
waters = settings.binding_site.waters
print(f'Number of waters in the binding site: {len(waters)}')
metal_ions = settings.binding_site.metals
print(f'Number of metal ions in the binding site: {len(metal_ions)}')
ligands = settings.binding_site.ligands
print(f'Number of ligands in the binding site: {len(ligands)}')


In [ ]:
#Alternatively, you can specify the binding site by using an atom or a residue(s). For example;
VAL717 = [r for r in protein.residues if r.identifier == 'A:VAL717'][0]
settings.binding_site = settings.BindingSiteFromResidue(protein, VAL717, 10.0)
atoms = settings.binding_site.atoms
print(f'Number of atoms in the binding site: {len(atoms)}')
residues = settings.binding_site.residues
print(f'Number of residues in the binding site: {len(residues)}')
waters = settings.binding_site.waters
print(f'Number of waters in the binding site: {len(waters)}')
metal_ions = settings.binding_site.metals
print(f'Number of metal ions in the binding site: {len(metal_ions)}')
ligands = settings.binding_site.ligands
print(f'Number of ligands in the binding site: {len(ligands)}')


### Ligand handling 

The rest of this notebook briefly demonstrates how the CSD Python API represents and processes **ligands**:

- Create a `ccdc.molecule.Molecule` from a SMILES string.
- Render a quick 2D depiction.
- Generate a small set of 3D **conformers**.

This is useful context for docking: GOLD ultimately docks **3D ligand conformations** into the binding site defined above.

In [ ]:
mol_a = Molecule.from_string("C=CC(=O)Nc1ccc2ncnc(Nc3ccc(F)c(Br)c3)c2c1")
print(type(mol_a))
lig_prep = Docker.LigandPreparation()

In [ ]:
from ccdc.diagram import DiagramGenerator
DiagramGenerator = DiagramGenerator()
img = DiagramGenerator.image(mol_a)
img

In [ ]:
from ccdc import io
from ccdc import conformer
conformer_gen = conformer.ConformerGenerator()
conformer_gen.settings.max_conformers = 10
conformers = conformer_gen.generate(mol_a)
